# 损失函数

In [1]:
import numpy as np

In [2]:
def mean_squared_error(y, t):
    """均方差"""
    return 0.5 * np.sum((y-t)**2)

def cross_entropy_error(y, t):
    """交叉熵误差"""
    delta = 1e-7
    return -np.sum(t * np.log(y + delta))


In [3]:
y1 = np.array([0.1, 0.05, 0.6, 0.0, 0.05, 0.1, 0.0, 0.1, 0.0, 0.0])
y2 = np.array([0.1, 0.05, 0.1, 0.0, 0.05, 0.1, 0.0, 0.6, 0.0, 0.0])
t = np.array([0, 0, 1, 0, 0, 0, 0, 0, 0, 0])

print(f"最接近正确答案的均方差: {mean_squared_error(y1, t)}")
print(f"最接近正确答案的交叉熵误差: {cross_entropy_error(y1, t)}")
print(f"最不接近正确答案的均方差: {mean_squared_error(y2, t)}")
print(f"最不接近正确答案的交叉熵误差: {cross_entropy_error(y2, t)}")


最接近正确答案的均方差: 0.09750000000000003
最接近正确答案的交叉熵误差: 0.510825457099338
最不接近正确答案的均方差: 0.5975
最不接近正确答案的交叉熵误差: 2.302584092994546


### mini-batch学习



In [4]:
import sys, os
import time
import numpy as np
import pickle


sys.path.append(os.pardir)
sys.path.append(os.curdir)
from dataset.mnist import load_mnist

(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, flatten=True, one_hot_label=True)
print(x_train.shape)
print(t_train.shape)
print(x_test.shape)
print(t_test.shape)


save_file: /Users/simeon/WorkSpace/Projects/Practice/RAGTest/test/../dataset/data/mnist.pkl
(60000, 784)
(60000, 10)
(10000, 784)
(10000, 10)


In [8]:
train_size = x_train.shape[0]
batch_size = 10
batch_mask = np.random.choice(train_size, batch_size)
x_batch = x_train[batch_mask]
t_batch = t_train[batch_mask]

print(batch_mask)
print(x_batch.shape)
print(t_batch.shape)


[ 8663 45752 13375 36782 22091 44826 47980 59596 11122 27722]
(10, 784)
(10, 10)


In [9]:
def cross_entropy_error_new(y, t):
    """交叉熵误差"""
    delta = 1e-7
    y = y.reshape(1, y.size)
    t = t.reshape(1, t.size)
    return -np.sum(t * np.log(y + delta))

In [26]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def softmax(x):
    c = np.max(x)
    exp_x = np.exp(x - c)
    sum_exp_x = np.sum(exp_x)
    return exp_x / sum_exp_x


def init_network():
    with open("../models/sample_weight.pkl", "rb") as f:
        network = pickle.load(f)

    return network


def predict(network, x):
    W1, W2, W3 = network['W1'], network['W2'], network['W3']
    b1, b2, b3 = network['b1'], network['b2'], network['b3']

    a1 = np.dot(x, W1) + b1
    z1 = sigmoid(a1)

    a2 = np.dot(z1, W2) + b2
    z2 = sigmoid(a2)

    a3 = np.dot(z2, W3) + b3
    y = softmax(a3)

    return y


network = init_network()
y0 = predict(network, x_batch[0])
true_number = np.argmax(t_batch[0])
print(f"正确答案是数字: {true_number}, 预测该数字的概率为: {y0[true_number]}")





正确答案是数字: 3, 预测该数字的概率为: 0.28293871879577637


In [22]:
res = predict(network, x_batch)
error = 0
error += cross_entropy_error(res, t_batch)
print(f"交叉熵误差为: {error / len(res)}")


交叉熵误差为: 3.5942597806453707
